# CIFAR-10 com MLP — Relatório de experimentos (grid search em blocos)

**Metodologia.** Em vez de escolher um valor por vez, cada bloco executa um **grid search** e o vencedor é
decidido pelos dados. O campeão de um bloco é lido automaticamente pelo bloco seguinte, sem nenhum valor escolhido à mão:

| Bloco | Pergunta | Grid |
|---|---|---|
| 0 — Referência | Onde partimos? | regressão logística e MLP do notebook original |
| 1 — Topologia | Qual estrutura base absorve melhor o CIFAR-10? | camadas × neurônios |
| 2 — Otimização | Qual algoritmo e taxa de aprendizagem fazem a rede campeã convergir melhor? | otimizador × lr |
| Checagem | A escolha em blocos se sustenta? | 2º e 3º do Bloco 1 com o otimizador campeão |
| 3 — Regularização e erro | Como dropout e função de erro afetam a generalização? | dropout × função de erro |
| Complementar A — Dropout longo | Os perdedores do Bloco 3 só precisavam de mais épocas? | dropout × função de erro com 100 épocas |
| Bônus — Data augmentation | Quanto um pré-processamento (crop + flip) acrescenta à rede campeã? | augmentation × dropout |

O notebook executa **o estudo completo da MLP em uma única execução** (Save & Run All): cada bloco herda o campeão do
anterior, e os complementares e o bônus partem do campeão mais recente.

**Protocolo de avaliação.**
- **Validação cruzada estratificada em 5 folds** sobre as 50.000 imagens de treino, com as mesmas partições em todos os experimentos (comparações pareadas).
- **Triagem:** cada configuração roda os folds 1–3; as 3 melhores **completam os folds 4–5** (sem refazer nada) e o campeão é a maior média nos 5 folds.
- **Escolha sempre pela validação.** O conjunto de teste (10.000 imagens) só é revelado na seção final, para os campeões. Usá-lo para escolher tornaria o número final otimista (vazamento de dados).
- Diferenças menores que o desvio padrão entre folds não devem ser tratadas como melhora real.

**Registro dos resultados.** Cada experimento grava em `outputs/{exp_name}/` (no Kaggle: `/kaggle/working/outputs`; no Colab: `/content/outputs`; no Jupyter local: a pasta do repositório):

| Arquivo | Conteúdo |
|---|---|
| `parametros.json` | hiperparâmetros exatos, arquitetura resolvida, versões e commit |
| `historico_treino.csv` | uma linha por **fold × época**: loss, acurácia, precision/recall/F1 (gerais e por classe) de treino e validação, gap |
| `historico_treino_agregado.csv` | média e desvio padrão entre folds, por época |
| `resultados.json` | resultado de cada fold e médias (validação na melhor época e teste) |
| `melhor_modelo.pth` | pesos com a menor `val/loss` (melhor fold); cada fold em `folds/fold_k/` |

Cada bloco grava em `outputs/_grids/{bloco}/`: `configs.csv`, `ranking_triagem.csv`, `ranking_final.csv`, `campeao.json` e `logs/`.
Tabelas e figuras dos relatórios ficam em `outputs/_relatorio/`. Tudo é gravado a cada época, antes do envio ao Weights & Biases (se houver chave; senão só o disco).

**Tempo estimado:** ~2 h com 1 GPU T4 ou **~1h–1h30 com 2× T4** (estimativa a partir de medições; varia ±30%).
Os experimentos são distribuídos entre as GPUs visíveis (2 por GPU, ajustável em `WORKERS_PER_GPU`). Sem GPU o código roda em CPU, mas o estudo deixa de ser praticável.

A célula seguinte descreve a **execução original no Kaggle** (como os resultados do relatório foram gerados) e como repetir o mesmo fluxo em Colab ou Jupyter local.

## Execução original no Kaggle (e como reproduzir)

Os números do relatório desta MLP saíram de um **Save Version → Save & Run All (Commit)** no Kaggle. Esse caminho **continua válido**: com os Settings abaixo, o notebook faz o mesmo estudo, nos mesmos blocos.

| Item | Como foi rodado |
|---|---|
| Acelerador | *Settings → Accelerator*: **GPU T4 ×2** |
| Internet | *Settings → Internet*: **On** (clone do GitHub, pip, W&B e fallback do CIFAR-10) |
| Dataset | *Add Input → Datasets*: **cifar10-python** (pasta `cifar-10-batches-py` ou arquivo `cifar-10-python.tar.gz`). Cópia conferida pelo MD5 oficial; sem ela o download de Toronto é muito lento |
| Código | clone de `https://github.com/diegoflyra/dfal-neural-networks.git` para `/tmp` (não vai para o Output) |
| `GITHUB_TOKEN` | *Add-ons → Secrets*, marcado como anexado. Obrigatório se o repositório for privado; o token nunca é impresso |
| `WANDB_API_KEY` | *Add-ons → Secrets*, anexado. **Opcional** — sem a chave o treino segue e tudo é gravado em disco; com ela cada fold vira uma run no W&B |
| Paralelismo | `WORKERS_PER_GPU = 2` (2 experimentos por GPU) |
| Saída | `/kaggle/working/outputs/` e `outputs.zip` na aba *Output* da versão |
| Duração | **~1h–1h30** com 2× T4 (~2 h com 1× T4; varia ±30%) |
| Retomada | anexar o Output da versão anterior como Input e preencher `RESUME_FROM`; folds já concluídos são pulados |

### No Colab, Jupyter local ou outra máquina

O **fluxo dos blocos é o mesmo** (referência → grids → campeão automático → teste só no final). Só o ambiente muda:

1. **GPU.** No Colab: *Runtime → Change runtime type → GPU* (em geral 1 T4, portanto mais lento). Ajuste `WORKERS_PER_GPU` se a memória apertar.
2. **Dataset.** Coloque `cifar-10-batches-py` ou `cifar-10-python.tar.gz` em `/content`, na pasta `data/` do repositório, ou em `CIFAR10_PATH`. Se não houver cópia, o notebook baixa do servidor oficial (MD5 conferido do mesmo jeito).
3. **Código.** Abra o notebook **dentro** de uma cópia do repositório (pastas `src/` e `grids/` no diretório atual) **ou** deixe clonar. Repo privado: defina `GITHUB_TOKEN` (Colab Secrets ou variável de ambiente). Repo público: clone sem chave.
4. **W&B.** Pode omitir. Sem `WANDB_API_KEY` o modo fica `disabled` e os CSV/JSON/PNG/`pth` saem em `outputs/` (`/content/outputs` no Colab; pasta do repositório no Jupyter local).
5. **Run All.** Igual ao Kaggle. Para retomar, aponte `RESUME_FROM` para a pasta `outputs` de uma execução anterior.

Não é preciso Kaggle, nem W&B, nem (com repo público ou cópia local) token do GitHub. O protocolo experimental não muda.

## 0. Preparação do ambiente

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/diegoflyra/dfal-neural-networks.git"
CLONE_DIR = "/tmp/dfal-neural-networks"


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(path):
    return os.path.isfile(os.path.join(path, "src", "grid_search.py")) and os.path.isdir(os.path.join(path, "grids"))


here = os.getcwd()
if _is_repo(here):
    REPO_DIR = here
    print(f"Código: cópia local em {REPO_DIR}")
else:
    REPO_DIR = CLONE_DIR
    token = _secret("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1) if token else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if token else "GitHub: clone sem token")
    if os.path.isdir(REPO_DIR) and _is_repo(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    else:
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        cloned = subprocess.run(["git", "clone", "-q", clone_url, REPO_DIR])
        if cloned.returncode != 0 or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "Falha ao clonar o repositório. Repositório privado exige GITHUB_TOKEN "
                "(Kaggle Secrets, Colab Secrets ou variável de ambiente), ou abra o notebook "
                "a partir de uma cópia local do projeto (pasta com src/ e grids/)."
            )
    print(f"Código: {REPO_DIR}")

os.chdir(REPO_DIR)
%cd {REPO_DIR}
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
import importlib.util
import subprocess
import sys

skip, pkgs = [], []
with open("requirements.txt", encoding="utf-8") as f:
    for line in f:
        pkg = line.strip()
        if not pkg or pkg.startswith("#"):
            continue
        name = pkg.split("==")[0].split(">=")[0].split("<=")[0].split("~=")[0].split("[")[0].strip().lower()
        if name in {"torch", "torchvision"} and importlib.util.find_spec(name) is not None:
            skip.append(name)
            continue
        pkgs.append(pkg)
if skip:
    print("Já instalado, não reinstalar (preserva CUDA do ambiente):", ", ".join(skip))
if pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import glob
import hashlib
import os
import shutil
import sys
import tarfile

import pandas as pd
from torchvision.datasets import CIFAR10

WORKERS_PER_GPU = 2  # experimentos simultâneos por GPU
RESUME_FROM = ""  # pasta outputs de uma execução anterior, para retomar
CIFAR10_PATH = ""  # pasta cifar-10-batches-py ou arquivo .tar.gz; vazio = procurar / baixar

# CIFAR-10: usa uma cópia local se o MD5 for o oficial (os mesmos hashes que o torchvision
# usa para validar https://www.cs.toronto.edu/~kriz/cifar.html). Caso contrário baixa o original.
DATA_DIR = os.path.join(REPO_DIR, "data")
CIFAR_DIR = os.path.join(DATA_DIR, "cifar-10-batches-py")
OFFICIAL_MD5 = dict(CIFAR10.train_list + CIFAR10.test_list + [[CIFAR10.meta["filename"], CIFAR10.meta["md5"]]])
os.makedirs(DATA_DIR, exist_ok=True)


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def cifar_is_official(folder):
    rows, ok = [], True
    for name, expected in OFFICIAL_MD5.items():
        path = os.path.join(folder, name)
        actual = md5(path) if os.path.isfile(path) else "arquivo ausente"
        ok &= actual == expected
        rows.append({"arquivo": name, "md5 oficial": expected, "md5 da cópia": actual,
                     "status": "OK" if actual == expected else "DIFERENTE"})
    display(pd.DataFrame(rows))
    return ok


def _unique(paths):
    seen, out = set(), []
    for path in paths:
        if path and os.path.exists(path) and path not in seen:
            seen.add(path)
            out.append(path)
    return out


def find_cifar():
    folders, archives = [], []
    explicit = CIFAR10_PATH or os.environ.get("CIFAR10_PATH", "").strip()
    roots = [explicit, "/kaggle/input", "/content", DATA_DIR, os.getcwd()]
    for root in roots:
        if not root or not os.path.exists(root):
            continue
        if os.path.isdir(root) and os.path.isfile(os.path.join(root, "data_batch_1")):
            folders.append(root)
        if os.path.isfile(root) and root.endswith((".tar.gz", ".tgz")):
            archives.append(root)
        if os.path.isdir(root):
            # /content/drive no Colab é grande demais para glob recursivo.
            if os.path.abspath(root) == "/content":
                for base in (root, os.path.join(root, "data")):
                    folders.append(os.path.join(base, "cifar-10-batches-py"))
                    archives.append(os.path.join(base, "cifar-10-python.tar.gz"))
            else:
                folders.extend(glob.glob(os.path.join(root, "**", "cifar-10-batches-py"), recursive=True))
                archives.extend(glob.glob(os.path.join(root, "**", "cifar-10-python.tar.gz"), recursive=True))
    return _unique(folders), _unique(archives)


if not os.path.isdir(CIFAR_DIR):
    folders, archives = find_cifar()
    if folders:
        print(f"Cópia encontrada: {folders[0]}")
        if os.path.abspath(folders[0]) != os.path.abspath(CIFAR_DIR):
            shutil.copytree(folders[0], CIFAR_DIR)
    elif archives:
        print(f"Arquivo encontrado: {archives[0]} | md5 {md5(archives[0])} (oficial: {CIFAR10.tgz_md5})")
        with tarfile.open(archives[0]) as tar:
            tar.extractall(DATA_DIR)
    else:
        print("CIFAR-10 não encontrado localmente; baixando do servidor oficial (pode levar vários minutos).")
        CIFAR10(root=DATA_DIR, train=True, download=True)
        CIFAR10(root=DATA_DIR, train=False, download=True)

if os.path.isdir(CIFAR_DIR):
    if cifar_is_official(CIFAR_DIR):
        print("CIFAR-10 verificado: todos os arquivos são idênticos aos oficiais.")
    else:
        shutil.rmtree(CIFAR_DIR)
        print("CÓPIA REJEITADA: arquivos diferentes dos oficiais. Baixando do servidor original.")
        CIFAR10(root=DATA_DIR, train=True, download=True)
        CIFAR10(root=DATA_DIR, train=False, download=True)

if os.path.isdir("/kaggle/working"):
    WORKDIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORKDIR = "/content"
else:
    WORKDIR = REPO_DIR

os.environ["EXP_OUTPUT_DIR"] = os.path.join(WORKDIR, "outputs")
os.makedirs(os.environ["EXP_OUTPUT_DIR"], exist_ok=True)
print(f"Resultados em {os.environ['EXP_OUTPUT_DIR']}")
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, os.environ["EXP_OUTPUT_DIR"], dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

wandb_key = _secret("WANDB_API_KEY")
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    print("W&B: chave carregada.")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado (sem chave). Resultados continuam em {os.environ['EXP_OUTPUT_DIR']}.")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import report_utils as rep

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino):
# GPUs, flags → arquitetura, carregamento em GPU/K-fold (baixa o CIFAR-10) e construção de todas as configurações dos grids
import shutil
if shutil.which("nvidia-smi"):
    !nvidia-smi -L
else:
    print("nvidia-smi não encontrado; o treino usa o device que o PyTorch enxergar (CPU se não houver GPU).")
!python tests/check_hyperparams.py
!python tests/check_data_loader.py
!python tests/check_grids.py

## Bloco 0 — Referências

Mesmo protocolo dos blocos seguintes (Adam, lr=1e-3, batch 128, early stopping com paciência 5, 5 folds), aplicado à
regressão logística (piso) e à MLP 64-128-64 do notebook original. Servem de ponto de comparação para todo o estudo.

Fixos no bloco: `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=40`, `patience=5`, `eval_train=True`, `activation=relu`, `loss_fn=cross_entropy`, `dropout=0.0`. Todas as configurações rodam os 5 folds.

In [ ]:
!python src/grid_search.py grids/mlp_b0_referencia.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
rep.grid_ranking("mlp_b0_referencia", "final")

## Bloco 1 — Topologia

**Pergunta:** qual combinação de profundidade (camadas ocultas) e largura (neurônios por camada) absorve melhor o CIFAR-10?

**Por que primeiro:** a topologia define a capacidade do modelo; otimização e regularização são ajustadas sobre ela.
Todas as redes usam a mesma otimização padrão e **nenhuma regularização**, com early stopping na `val/loss`
para que redes grandes não sejam penalizadas por treinar além do ponto ótimo.

| Eixo | Valores |
|---|---|
| `mlp_layers` | `1`, `2`, `3`, `4`, `5` |
| `mlp_neurons` | `128`, `256`, `512`, `1024`, `2048` |

**25 combinações.** Fixos no bloco: `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=40`, `patience=5`, `eval_train=True`, `activation=relu`, `loss_fn=cross_entropy`, `dropout=0.0`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds.

**O que observar:** o heatmap camadas × neurônios, o `gap/accuracy` (overfitting) das redes maiores e se ganhos de
capacidade continuam aparecendo na validação.

In [ ]:
!python src/grid_search.py grids/mlp_b1_topologia.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `mlp_b1_topologia`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("mlp_b1_topologia", "triagem")

In [ ]:
rep.heatmap("mlp_b1_topologia", row="mlp_layers", col="mlp_neurons")

In [ ]:
rep.plot_grid_bars("mlp_b1_topologia")

#### Confirmação das finalistas e campeão — `mlp_b1_topologia`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("mlp_b1_topologia", "final")

In [ ]:
rep.show_champion("mlp_b1_topologia")
rep.plot_finalists("mlp_b1_topologia")

In [ ]:
rep.heatmap("mlp_b1_topologia", row="mlp_layers", col="mlp_neurons", value="gap/accuracy_mean")

### 📝 Análise — Bloco 1

- **Profundidade ou largura: o que mais contribuiu?** _…_
- **A partir de qual tamanho a validação estagna?** _…_
- **Como o gap cresce com a capacidade?** _…_
- **Diferença do campeão para as referências (validação):** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/)
!cd {WORKDIR} && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bloco 2 — Otimização

**Pergunta:** com a topologia campeã do Bloco 1 fixa, qual combinação de algoritmo e taxa de aprendizagem converge melhor?

**Base:** o campeão do Bloco 1 (abaixo), lido de `outputs/_grids/mlp_b1_topologia/campeao.json`. SGD usa momentum 0,9.
O grid de lr é comum aos dois algoritmos de propósito: ele mostra a faixa em que cada um funciona (lr baixos
não convergem no SGD; lr altos podem divergir no Adam — divergências são registradas sem perda de arquivos).

| Eixo | Valores |
|---|---|
| `optimizer` | `sgd`, `adam` |
| `lr` | `0.0001`, `0.0003`, `0.001`, `0.003`, `0.01`, `0.03`, `0.1` |

**14 combinações.** Herdado do campeão de: `mlp_b1_topologia`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds.

**O que observar:** o heatmap otimizador × lr, a `melhor_epoca_media` (velocidade) e as curvas das finalistas.

In [ ]:
rep.show_champion("mlp_b1_topologia")

In [ ]:
!python src/grid_search.py grids/mlp_b2_otimizacao.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `mlp_b2_otimizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("mlp_b2_otimizacao", "triagem")

In [ ]:
rep.heatmap("mlp_b2_otimizacao", row="optimizer", col="lr")

In [ ]:
rep.plot_grid_bars("mlp_b2_otimizacao")

#### Confirmação das finalistas e campeão — `mlp_b2_otimizacao`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("mlp_b2_otimizacao", "final")

In [ ]:
rep.show_champion("mlp_b2_otimizacao")
rep.plot_finalists("mlp_b2_otimizacao")

In [ ]:
rep.heatmap("mlp_b2_otimizacao", row="optimizer", col="lr", value="melhor_epoca_media")

### 📝 Análise — Bloco 2

- **Faixa de lr útil para SGD e para Adam:** _…_
- **Qual convergiu em menos épocas?** _…_
- **Houve divergência? Em quais combinações?** _…_
- **Ganho sobre o Bloco 1 (validação):** _…_

### Checagem de interação

A busca em blocos assume que a melhor topologia com Adam lr=1e-3 continua sendo a melhor com o otimizador campeão.
Para verificar, o 2º e o 3º colocados do Bloco 1 são treinados com o otimizador/lr campeões do Bloco 2 (5 folds).
O Bloco 3 parte da melhor rede entre o campeão do Bloco 2 e esta checagem.

Herdado do campeão de: `mlp_b2_otimizacao`. Todas as configurações rodam os 5 folds.

In [ ]:
!python src/grid_search.py grids/mlp_b2_checagem.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
pd.concat([rep.grid_ranking("mlp_b2_otimizacao", "final").head(1),
           rep.grid_ranking("mlp_b2_checagem", "final")], ignore_index=True)

### 📝 Análise — Checagem

- **A ordem das topologias se manteve com o novo otimizador?** _…_
- **Se mudou, o que isso indica sobre a interação topologia × otimização?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/)
!cd {WORKDIR} && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bloco 3 — Regularização e função de erro

**Pergunta:** com a rede convergindo bem, como a taxa de dropout e a função de erro afetam a generalização?

**Base:** a melhor rede entre o campeão do Bloco 2 e a checagem. Épocas e paciência aumentadas, pois dropout
retarda a convergência. A seleção é pela `val/accuracy`: a `val/loss` de MSE e de entropia cruzada estão em
escalas diferentes e não são comparáveis entre si.

| Eixo | Valores |
|---|---|
| `dropout` | `0.0`, `0.2`, `0.3`, `0.5` |
| `loss_fn` | `cross_entropy`, `mse` |

**8 combinações.** Herdado do campeão de: `mlp_b2_otimizacao`, `mlp_b2_checagem`. Fixos no bloco: `epochs=50`, `patience=7`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds.

**O que observar:** o `gap/accuracy` diminuindo com dropout, o ponto em que dropout passa a causar subajuste
e a diferença de convergência entre MSE e entropia cruzada.

In [ ]:
!python src/grid_search.py grids/mlp_b3_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `mlp_b3_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("mlp_b3_regularizacao", "triagem")

In [ ]:
rep.heatmap("mlp_b3_regularizacao", row="dropout", col="loss_fn")

In [ ]:
rep.plot_grid_bars("mlp_b3_regularizacao")

#### Confirmação das finalistas e campeão — `mlp_b3_regularizacao`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("mlp_b3_regularizacao", "final")

In [ ]:
rep.show_champion("mlp_b3_regularizacao")
rep.plot_finalists("mlp_b3_regularizacao")

In [ ]:
rep.heatmap("mlp_b3_regularizacao", row="dropout", col="loss_fn", value="gap/accuracy_mean")

### 📝 Análise — Bloco 3

- **Dropout reduziu o gap? A partir de qual taxa houve subajuste?** _…_
- **MSE vs entropia cruzada: diferença de acurácia e de velocidade de convergência:** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/)
!cd {WORKDIR} && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Complementar A — Dropout com orçamento maior

**Pergunta:** configurações que atingiram a melhor época perto do limite de 50 épocas no Bloco 3 (tipicamente dropout alto)
mudariam de posição com mais tempo? A vantagem da entropia cruzada sobre MSE se mantém quando as duas convergem?

**Base:** o campeão do Bloco 3, com **100 épocas e paciência 10**. A configuração idêntica ao campeão é re-treinada no
novo orçamento para uma comparação justa.

| Eixo | Valores |
|---|---|
| `dropout` | `0.2`, `0.5` |
| `loss_fn` | `cross_entropy`, `mse` |

**4 combinações.** Herdado do campeão de: `mlp_b3_regularizacao`. Fixos no bloco: `epochs=100`, `patience=10`. Todas as configurações rodam os 5 folds.

**O que observar:** `melhor_epoca_media` (agora com folga até 100) e a comparação **pareada por fold** contra a receita campeã.

In [ ]:
!python src/grid_search.py grids/mlp_b3b_dropout_longo.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
rep.grid_ranking("mlp_b3b_dropout_longo", "final")

In [ ]:
rep.heatmap("mlp_b3b_dropout_longo", row="dropout", col="loss_fn")

In [ ]:
rep.heatmap("mlp_b3b_dropout_longo", row="dropout", col="loss_fn", value="melhor_epoca_media")

In [ ]:
rep.heatmap("mlp_b3b_dropout_longo", row="dropout", col="loss_fn", value="gap/accuracy_mean")

In [ ]:
rep.show_champion("mlp_b3b_dropout_longo")
rep.plot_finalists("mlp_b3b_dropout_longo")

#### Comparação pareada por fold

Referência: a configuração idêntica à receita campeã do Bloco 3. Cada ponto é a diferença em um fold; “(4/5)” indica em quantos folds a configuração venceu a referência.

In [ ]:
rep.paired_comparison(rep.base_config_name("mlp_b3b_dropout_longo"), "mlp_b3b_dropout_longo__*")

### 📝 Análise — Complementar A

- **Quantas épocas as configurações precisaram com o novo limite?** _…_
- **Alguma configuração mudou de posição em relação ao Bloco 3?** _…_
- **Entropia cruzada vs MSE fold a fold:** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/)
!cd {WORKDIR} && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bônus — Data augmentation

Data augmentation é um **pré-processamento**, não um hiperparâmetro da rede: por isso fica fora da sequência principal
de blocos. Cria variações plausíveis de cada imagem de treino a cada época, aumentando a diversidade efetiva dos dados.

**Transformações** (apenas no treino, direto na GPU; validação e teste usam as imagens originais):
**RandomCrop 32 com padding 4** (desloca até 4 pixels) e **RandomHorizontalFlip** (espelha com probabilidade 0,5).

**Hipótese.** Sem noção de vizinhança entre pixels, cada deslocamento é uma entrada nova para a MLP: o augmentation pode ensinar alguma invariância, mas o ganho deve ser pequeno e a convergência lenta.

**Base:** o campeão de `mlp_b3b_dropout_longo`, com mais épocas e paciência, pois augmentation retarda a convergência. A receita
campeã **sem** augmentation é re-treinada no mesmo bloco, como
referência pareada.

| Eixo | Valores |
|---|---|
| `augment` | `False`, `True` |
| `dropout` | `0.0`, `0.2` |

**4 combinações.** Herdado do campeão de: `mlp_b3b_dropout_longo`. Fixos no bloco: `epochs=150`, `patience=15`. Todas as configurações rodam os 5 folds.

**O que observar:** o ganho pareado sobre a referência sem augmentation, a queda do `gap/accuracy` e se a
`melhor_epoca_media` ficou perto do limite de épocas (resultado possivelmente limitado pelo orçamento).

In [ ]:
!python src/grid_search.py grids/mlp_b4_augmentation.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
rep.heatmap("mlp_b4_augmentation", row="dropout", col="augment")

In [ ]:
rep.heatmap("mlp_b4_augmentation", row="dropout", col="augment", value="gap/accuracy_mean")

In [ ]:
rep.heatmap("mlp_b4_augmentation", row="dropout", col="augment", value="melhor_epoca_media")

In [ ]:
rep.grid_ranking("mlp_b4_augmentation", "final")

In [ ]:
rep.show_champion("mlp_b4_augmentation")
rep.plot_finalists("mlp_b4_augmentation")

#### Comparação pareada por fold (validação)

Referência: a receita campeã sem augmentation, treinada neste mesmo bloco.

In [ ]:
rep.paired_comparison(rep.base_config_name("mlp_b4_augmentation"), "mlp_b4_augmentation__*")

### 📝 Análise — Bônus

- **Ganho do augmentation na validação (pareado):** _…_
- **O gap treino-validação caiu?** _…_
- **Com augmentation, o dropout ainda é necessário?** _…_
- **Algum resultado ficou limitado pelo orçamento de épocas?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/)
!cd {WORKDIR} && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Resultado final — teste revelado

Até aqui todas as escolhas foram feitas pela validação cruzada. Agora o conjunto de teste é usado **uma única vez**
para os campeões de cada bloco e para as referências do Bloco 0. A tabela mostra média ± desvio do teste entre os
5 modelos (um por fold) de cada configuração. O campeão da sequência principal é o de `mlp_b3b_dropout_longo`; o bônus mostra
quanto o augmentation acrescenta sobre ele.

In [ ]:
final = rep.final_report(['mlp_b0_referencia', 'mlp_b1_topologia', 'mlp_b2_otimizacao', 'mlp_b2_checagem', 'mlp_b3_regularizacao', 'mlp_b3b_dropout_longo', 'mlp_b4_augmentation'])
final

Desempenho por classe no teste: referências do Bloco 0 × campeão principal × campeão com augmentation. Classes com baixo recall indicam confusões sistemáticas (veja a matriz de confusão).

In [ ]:
campeao_principal = rep.champion_name("mlp_b3b_dropout_longo")
campeao_bonus = rep.champion_name("mlp_b4_augmentation")
comparar = list(final.loc[final["papel"] == "referência", "exp_name"]) + [campeao_principal, campeao_bonus]
rep.plot_per_class(comparar, metric="recall")
rep.plot_per_class(comparar, metric="precision")
pd.read_csv(os.path.join(os.environ["EXP_OUTPUT_DIR"], campeao_principal, "matriz_confusao_teste.csv"), index_col=0)

#### Métricas gerais e por classe dos campeões

Para cada campeão: **acurácia, precision, recall e F1 gerais** (macro, no topo da figura) e **precision, recall e F1 de cada classe**
(tabela), no teste, em média ± desvio entre os 5 modelos (um por fold). Por classe, a acurácia é a fração das imagens daquela
classe classificadas corretamente, igual ao recall. Campeões do Bloco 3 (regularização) e do Bônus (augmentation).
Figura e CSV: `outputs/_relatorio/metricas_por_classe_test_<experimento>.*`.

In [ ]:
rep.champion_class_metrics("mlp_b3_regularizacao")

In [ ]:
rep.champion_class_metrics("mlp_b4_augmentation")

In [ ]:
!cd {WORKDIR} && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## 📝 Conclusões

- **Estrutura base que melhor absorveu o CIFAR-10 (Bloco 1) e por quê:** _…_
- **Ganho de otimização (Bloco 2) e sensibilidade à taxa de aprendizagem:** _…_
- **A checagem confirmou a escolha em blocos?** _…_
- **Efeito da regularização/erro (Bloco 3):** _…_
- **O que o experimento complementar corrigiu ou confirmou:** _…_
- **Ganho total sobre a referência (teste):** _…_
- **Bônus — quanto o augmentation acrescentou e por quê:** _…_
- **Classes mais difíceis e hipótese:** _…_